# Prueba Módulo 3 — Caso de Estudio *UrbanRent Analytics*

Pipeline completo de análisis con Spark sobre datos de reservas, apartamentos y propietarios.
Resolución de las **24 preguntas** distribuidas en 5 partes:

1. Carga y exploración inicial (P1-P3)
2. Operaciones básicas (P4-P7)
3. Agregaciones y métricas (P8-P12)
4. Joins (P13-P18)
5. Funciones integradas y UDFs (P19-P24)


## ⚙️ Celda 0 — Inicialización del entorno

In [1]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("UrbanRent_Analytics")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.sql.crossJoin.enabled", "true")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

import spark.implicits._
spark.sparkContext.setLogLevel("ERROR")

println(s"✅ UrbanRent Analytics iniciado — Spark ${spark.version} · Scala ${scala.util.Properties.versionString}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/04 10:08:14 INFO SparkContext: Running Spark version 4.1.1
26/05/04 10:08:15 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/05/04 10:08:15 INFO SparkContext: Java version 17.0.18+8
26/05/04 10:08:17 INFO ResourceUtils: ==============================================================
26/05/04 10:08:17 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/04 10:08:17 INFO ResourceUtils: ==============================================================
26/05/04 10:08:17 INFO SparkContext: Submitted application: UrbanRent_Analytics
26/05/04 10:08:17 INFO SecurityManager: Changing view acls to: gre
26/05/04 10:08:17 INFO SecurityManager: Changing modify acls to: gre
26/05/04 10:08:17 INFO SecurityManager: Changing view acls groups to: gre
26/05/04 10:08:17 INFO SecurityManager: Changing modify acls groups to: gre
26/05/04 10:08:17 INFO SecurityManager: SecurityManager: authenticatio

✅ UrbanRent Analytics iniciado — Spark 4.1.1 · Scala version 2.13.17


import $ivy.$
import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@d8cc0c7
import spark.implicits._

### Generar los ficheros del caso (CSV + JSON) en `data/`

Esta celda escribe los tres ficheros en disco para poder cargarlos después con la API de lectura de Spark.


In [2]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val dataDir = Paths.get("data")
if (!Files.exists(dataDir)) Files.createDirectories(dataDir)

val reservasCsv =
  """id_reserva,id_apartamento,fecha_entrada,fecha_salida,num_huespedes,precio_noche,canal,valoracion,ciudad
    |R001,APT001,2025-01-10,2025-01-13,2,95.0,Airbnb,4.8,Madrid
    |R002,APT002,2025-01-12,2025-01-15,4,120.0,Booking,4.5,Barcelona
    |R003,APT003,2025-01-20,2025-01-22,1,75.0,Directo,5.0,Valencia
    |R004,APT001,2025-02-01,2025-02-05,3,95.0,Airbnb,4.7,Madrid
    |R005,APT004,2025-02-10,2025-02-12,2,110.0,Booking,4.2,Sevilla
    |R006,APT002,2025-02-14,2025-02-17,5,120.0,Airbnb,4.9,Barcelona
    |R007,APT005,2025-02-20,2025-02-23,2,85.0,Directo,4.6,Bilbao
    |R008,APT003,2025-03-01,2025-03-04,1,75.0,Booking,4.4,Valencia
    |R009,APT001,2025-03-10,2025-03-14,4,95.0,Airbnb,4.8,Madrid
    |R010,APT006,2025-03-15,2025-03-17,2,130.0,Directo,5.0,Madrid
    |R011,APT004,2025-03-20,2025-03-22,3,110.0,Airbnb,4.3,Sevilla
    |R012,APT007,2025-04-01,2025-04-04,2,90.0,Booking,4.5,Barcelona
    |R013,APT005,2025-04-08,2025-04-10,1,85.0,Directo,4.7,Bilbao
    |R014,APT006,2025-04-12,2025-04-16,3,130.0,Airbnb,4.9,Madrid
    |R015,APT002,2025-04-20,2025-04-23,4,120.0,Booking,4.6,Barcelona
    |R016,APT008,2025-05-01,2025-05-04,2,70.0,Directo,4.1,Valencia
    |R017,APT003,2025-05-10,2025-05-13,2,75.0,Airbnb,4.5,Valencia
    |R018,APT007,2025-05-15,2025-05-18,3,90.0,Booking,4.8,Barcelona
    |R019,APT001,2025-05-20,2025-05-24,2,95.0,Directo,5.0,Madrid
    |R020,APT009,2025-05-25,2025-05-28,1,65.0,Airbnb,3.9,Bilbao
    |R021,APT006,2025-06-01,2025-06-05,4,130.0,Booking,4.7,Madrid
    |R022,APT010,2025-06-08,2025-06-10,2,80.0,Directo,4.3,Sevilla
    |R023,APT004,2025-06-15,2025-06-18,3,110.0,Airbnb,4.6,Sevilla
    |R024,APT008,2025-06-20,2025-06-22,1,70.0,Booking,4.2,Valencia
    |R025,APT005,2025-06-25,2025-06-28,2,85.0,Directo,4.8,Bilbao""".stripMargin

val apartamentosCsv =
  """id_apartamento,tipo,habitaciones,metros_cuadrados,tiene_parking,propietario_id,activo
    |APT001,Estudio,1,35,false,P01,true
    |APT002,Piso,3,85,true,P02,true
    |APT003,Estudio,1,30,false,P01,true
    |APT004,Piso,2,65,true,P03,true
    |APT005,Ático,2,70,true,P04,true
    |APT006,Piso,3,90,true,P02,true
    |APT007,Estudio,1,28,false,P05,true
    |APT008,Estudio,1,32,false,P03,false
    |APT009,Piso,2,55,false,P04,true
    |APT010,Ático,3,100,true,P05,true""".stripMargin

val propietariosJson =
  """[
    |  {"propietario_id": "P01", "nombre": "Laura Sánchez", "ciudad": "Madrid", "antiguedad_anios": 5, "tipo_contrato": "Premium", "especialidades": ["Estudio","Piso"]},
    |  {"propietario_id": "P02", "nombre": "Carlos Méndez", "ciudad": "Barcelona", "antiguedad_anios": 3, "tipo_contrato": "Estándar", "especialidades": ["Piso"]},
    |  {"propietario_id": "P03", "nombre": "Ana Ferrero", "ciudad": "Sevilla", "antiguedad_anios": 7, "tipo_contrato": "Premium", "especialidades": ["Piso","Ático"]},
    |  {"propietario_id": "P04", "nombre": "Javier Ruiz", "ciudad": "Bilbao", "antiguedad_anios": 2, "tipo_contrato": "Estándar", "especialidades": ["Ático","Piso"]},
    |  {"propietario_id": "P05", "nombre": "Marta Gil", "ciudad": "Valencia", "antiguedad_anios": 6, "tipo_contrato": "Premium", "especialidades": ["Estudio","Ático"]}
    |]""".stripMargin

Files.write(dataDir.resolve("reservas.csv"),      reservasCsv.getBytes(StandardCharsets.UTF_8))
Files.write(dataDir.resolve("apartamentos.csv"),  apartamentosCsv.getBytes(StandardCharsets.UTF_8))
Files.write(dataDir.resolve("propietarios.json"), propietariosJson.getBytes(StandardCharsets.UTF_8))

println("✅ Ficheros creados en ./data/:")
Files.list(dataDir).forEach(p => println(s"  - ${p.getFileName}"))

✅ Ficheros creados en ./data/:
  - apartamentos.csv
  - propietarios.json
  - reservas.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
dataDir: java.nio.file.Path = data
res2_3: Any = ()
reservasCsv: String = """id_reserva,id_apartamento,fecha_entrada,fecha_salida,num_huespedes,precio_noche,canal,valoracion,ciudad
R001,APT001,2025-01-10,2025-01-13,2,95.0,Airbnb,4.8,Madrid
R002,APT002,2025-01-12,2025-01-15,4,120.0,Booking,4.5,Barcelona
R003,APT003,2025-01-20,2025-01-22,1,75.0,Directo,5.0,Valencia
R004,APT001,2025-02-01,2025-02-05,3,95.0,Airbnb,4.7,Madrid
R005,APT004,2025-02-10,2025-02-12,2,110.0,Booking,4.2,Sevilla
R006,APT002,2025-02-14,2025-02-17,5,120.0,Airbnb,4.9,Barcelona
R007,APT005,2025-02-20,2025-02-23,2,85.0,Directo,4.6,Bilbao
R008,APT003,2025-03-01,2025-03-04,1,75.0,Booking,4.4,Valencia
R009,APT001,2025-03-10,2025-03-14,4,95.0,Airbnb,4.8,Madrid
R010,APT006,2025-03-15,2025-03-17,2,130.0,Directo,5.0,Madrid
R011,APT004,2025-03-20,2025-03-22,3,110.0,Airbnb,4.3,Sevilla
R012,APT007,2025-04-01,2025-04-04,2,90.0,Booking,4.5,Barcelona
R013,AP

---

# 🗂️ PARTE 1 — Carga y exploración inicial


## Pregunta 1 — ¿Están bien formateados nuestros datos de reservas?

In [3]:
val reservasInfer = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("data/reservas.csv")

println("=== Primeras 8 reservas ===")
reservasInfer.show(8, truncate = false)

println("=== Schema inferido ===")
reservasInfer.printSchema()

println("=== Estadísticas descriptivas ===")
reservasInfer.describe("precio_noche", "num_huespedes", "valoracion").show()

println(s"Total de reservas: ${reservasInfer.count()}")
println("Columnas: " + reservasInfer.columns.mkString(" | "))

=== Primeras 8 reservas ===
+----------+--------------+-------------+------------+-------------+------------+-------+----------+---------+
|id_reserva|id_apartamento|fecha_entrada|fecha_salida|num_huespedes|precio_noche|canal  |valoracion|ciudad   |
+----------+--------------+-------------+------------+-------------+------------+-------+----------+---------+
|R001      |APT001        |2025-01-10   |2025-01-13  |2            |95.0        |Airbnb |4.8       |Madrid   |
|R002      |APT002        |2025-01-12   |2025-01-15  |4            |120.0       |Booking|4.5       |Barcelona|
|R003      |APT003        |2025-01-20   |2025-01-22  |1            |75.0        |Directo|5.0       |Valencia |
|R004      |APT001        |2025-02-01   |2025-02-05  |3            |95.0        |Airbnb |4.7       |Madrid   |
|R005      |APT004        |2025-02-10   |2025-02-12  |2            |110.0       |Booking|4.2       |Sevilla  |
|R006      |APT002        |2025-02-14   |2025-02-17  |5            |120.0       |Air

reservasInfer: org.apache.spark.sql.package.DataFrame = [id_reserva: string, id_apartamento: string ... 7 more fields]

## Pregunta 2 — ¿Es correcto el schema inferido o necesitamos ajustarlo?

Definimos un `StructType` manual con `DateType` para las fechas y los tipos numéricos correctos.


In [4]:
val reservasSchema = StructType(Array(
  StructField("id_reserva",     StringType,  nullable = true),
  StructField("id_apartamento", StringType,  nullable = true),
  StructField("fecha_entrada",  DateType,    nullable = true),
  StructField("fecha_salida",   DateType,    nullable = true),
  StructField("num_huespedes",  IntegerType, nullable = true),
  StructField("precio_noche",   DoubleType,  nullable = true),
  StructField("canal",          StringType,  nullable = true),
  StructField("valoracion",     DoubleType,  nullable = true),
  StructField("ciudad",         StringType,  nullable = true)
))

val reservas = spark.read
  .option("header", "true")
  .option("dateFormat", "yyyy-MM-dd")
  .schema(reservasSchema)
  .csv("data/reservas.csv")

println("=== Schema manual aplicado ===")
reservas.printSchema()
reservas.show(5, truncate = false)

=== Schema manual aplicado ===
root
 |-- id_reserva: string (nullable = true)
 |-- id_apartamento: string (nullable = true)
 |-- fecha_entrada: date (nullable = true)
 |-- fecha_salida: date (nullable = true)
 |-- num_huespedes: integer (nullable = true)
 |-- precio_noche: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- valoracion: double (nullable = true)
 |-- ciudad: string (nullable = true)

+----------+--------------+-------------+------------+-------------+------------+-------+----------+---------+
|id_reserva|id_apartamento|fecha_entrada|fecha_salida|num_huespedes|precio_noche|canal  |valoracion|ciudad   |
+----------+--------------+-------------+------------+-------------+------------+-------+----------+---------+
|R001      |APT001        |2025-01-10   |2025-01-13  |2            |95.0        |Airbnb |4.8       |Madrid   |
|R002      |APT002        |2025-01-12   |2025-01-15  |4            |120.0       |Booking|4.5       |Barcelona|
|R003      |APT003        |2

reservasSchema: StructType = Seq(
  StructField(
    name = "id_reserva",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "id_apartamento",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "fecha_entrada",
    dataType = DateType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "fecha_salida",
    dataType = DateType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "num_huespedes",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "precio_noche",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "canal",
...
reservas: org.apache.spark.sql.package.DataFrame = [id_reserva: string, id_apartamento: string ... 7 more fields]

## Pregunta 3 — ¿Cuántos apartamentos tenemos y cuáles son sus características?

In [5]:
val apartamentos = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("data/apartamentos.csv")

val propietarios = spark.read
  .option("multiline", "true")
  .json("data/propietarios.json")

println("=== APARTAMENTOS ===")
apartamentos.printSchema()
apartamentos.show(truncate = false)
println(s"Total apartamentos: ${apartamentos.count()}")
println(s"Apartamentos activos: ${apartamentos.filter($"activo" === true).count()}")

println("\n=== PROPIETARIOS ===")
propietarios.printSchema()
propietarios.show(truncate = false)
println(s"Total propietarios: ${propietarios.count()}")

=== APARTAMENTOS ===
root
 |-- id_apartamento: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- habitaciones: integer (nullable = true)
 |-- metros_cuadrados: integer (nullable = true)
 |-- tiene_parking: boolean (nullable = true)
 |-- propietario_id: string (nullable = true)
 |-- activo: boolean (nullable = true)

+--------------+-------+------------+----------------+-------------+--------------+------+
|id_apartamento|tipo   |habitaciones|metros_cuadrados|tiene_parking|propietario_id|activo|
+--------------+-------+------------+----------------+-------------+--------------+------+
|APT001        |Estudio|1           |35              |false        |P01           |true  |
|APT002        |Piso   |3           |85              |true         |P02           |true  |
|APT003        |Estudio|1           |30              |false        |P01           |true  |
|APT004        |Piso   |2           |65              |true         |P03           |true  |
|APT005        |Ático  |2    

apartamentos: org.apache.spark.sql.package.DataFrame = [id_apartamento: string, tipo: string ... 5 more fields]
propietarios: org.apache.spark.sql.package.DataFrame = [antiguedad_anios: bigint, ciudad: string ... 4 more fields]

---

# 🔧 PARTE 2 — Operaciones básicas sobre DataFrames


## Pregunta 4 — Reservas de más de 3 noches

In [6]:
val reservasConMetricas = reservas
  .withColumn("num_noches",      datediff($"fecha_salida", $"fecha_entrada"))
  .withColumn("ingreso_reserva", $"num_noches" * $"precio_noche")

val reservasLargas = reservasConMetricas
  .filter($"num_noches" > 3)
  .select("id_reserva", "ciudad", "canal", "num_noches", "ingreso_reserva")
  .orderBy($"ingreso_reserva".desc)

reservasLargas.show(truncate = false)
println(s"Reservas largas (> 3 noches): ${reservasLargas.count()}")

+----------+------+-------+----------+---------------+
|id_reserva|ciudad|canal  |num_noches|ingreso_reserva|
+----------+------+-------+----------+---------------+
|R014      |Madrid|Airbnb |4         |520.0          |
|R021      |Madrid|Booking|4         |520.0          |
|R004      |Madrid|Airbnb |4         |380.0          |
|R009      |Madrid|Airbnb |4         |380.0          |
|R019      |Madrid|Directo|4         |380.0          |
+----------+------+-------+----------+---------------+

Reservas largas (> 3 noches): 5


reservasConMetricas: org.apache.spark.sql.package.DataFrame = [id_reserva: string, id_apartamento: string ... 9 more fields]
reservasLargas: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [id_reserva: string, ciudad: string ... 3 more fields]

## Pregunta 5 — Canales activos y precio medio por canal

In [7]:
val canalesUnicos = reservas
  .dropDuplicates("canal")
  .select("canal", "precio_noche")
  .orderBy($"precio_noche".desc)

canalesUnicos.show(truncate = false)

+-------+------------+
|canal  |precio_noche|
+-------+------------+
|Booking|120.0       |
|Airbnb |95.0        |
|Directo|75.0        |
+-------+------------+



canalesUnicos: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [canal: string, precio_noche: double]

## Pregunta 6 — Suplemento del 10% para reservas de más de 2 huéspedes

In [8]:
val reservasConSuplemento = reservasConMetricas
  .withColumn("precio_con_suplemento",
    round(when($"num_huespedes" > 2, $"precio_noche" * 1.10).otherwise($"precio_noche"), 2))
  .withColumn("ingreso_estimado", round($"num_noches" * $"precio_con_suplemento", 2))
  .select("id_reserva", "ciudad", "num_huespedes", "precio_noche", "precio_con_suplemento", "ingreso_estimado")
  .orderBy($"ingreso_estimado".desc)

reservasConSuplemento.show(truncate = false)

+----------+---------+-------------+------------+---------------------+----------------+
|id_reserva|ciudad   |num_huespedes|precio_noche|precio_con_suplemento|ingreso_estimado|
+----------+---------+-------------+------------+---------------------+----------------+
|R014      |Madrid   |3            |130.0       |143.0                |572.0           |
|R021      |Madrid   |4            |130.0       |143.0                |572.0           |
|R004      |Madrid   |3            |95.0        |104.5                |418.0           |
|R009      |Madrid   |4            |95.0        |104.5                |418.0           |
|R002      |Barcelona|4            |120.0       |132.0                |396.0           |
|R006      |Barcelona|5            |120.0       |132.0                |396.0           |
|R015      |Barcelona|4            |120.0       |132.0                |396.0           |
|R019      |Madrid   |2            |95.0        |95.0                 |380.0           |
|R023      |Sevilla  

reservasConSuplemento: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [id_reserva: string, ciudad: string ... 4 more fields]

## Pregunta 7 — Ciudades en `apartamentos`/`propietarios` que no aparecen en `reservas`

In [9]:
val propietariosRen = propietarios.withColumnRenamed("ciudad", "ciudad_propietario")

val ciudadesReservas    = reservas.select("ciudad").distinct().orderBy("ciudad")
val ciudadesPropietarios = propietarios.select("ciudad").distinct().orderBy("ciudad")

println("=== Ciudades en reservas ===")
ciudadesReservas.show()
println("=== Ciudades en propietarios ===")
ciudadesPropietarios.show()

println("=== Diferencia (en propietarios pero no en reservas) ===")
ciudadesPropietarios.except(ciudadesReservas).show()

=== Ciudades en reservas ===
+---------+
|   ciudad|
+---------+
|Barcelona|
|   Bilbao|
|   Madrid|
|  Sevilla|
| Valencia|
+---------+

=== Ciudades en propietarios ===
+---------+
|   ciudad|
+---------+
|Barcelona|
|   Bilbao|
|   Madrid|
|  Sevilla|
| Valencia|
+---------+

=== Diferencia (en propietarios pero no en reservas) ===
+------+
|ciudad|
+------+
+------+



propietariosRen: org.apache.spark.sql.package.DataFrame = [antiguedad_anios: bigint, ciudad_propietario: string ... 4 more fields]
ciudadesReservas: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string]
ciudadesPropietarios: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string]

---

# 📊 PARTE 3 — Agregaciones y métricas de negocio


## Pregunta 8 — Ciudad con más ingresos

In [10]:
val ingresosPorCiudad = reservasConMetricas
  .groupBy("ciudad")
  .agg(
    count("*").alias("num_reservas"),
    round(sum("ingreso_reserva"), 2).alias("ingreso_total"),
    round(avg("ingreso_reserva"), 2).alias("ticket_medio"),
    round(avg("valoracion"), 2).alias("valoracion_media")
  )
  .orderBy($"ingreso_total".desc)

ingresosPorCiudad.show(truncate = false)

+---------+------------+-------------+------------+----------------+
|ciudad   |num_reservas|ingreso_total|ticket_medio|valoracion_media|
+---------+------------+-------------+------------+----------------+
|Madrid   |7           |2725.0       |389.29      |4.84            |
|Barcelona|5           |1620.0       |324.0       |4.66            |
|Valencia |5           |950.0        |190.0       |4.44            |
|Sevilla  |4           |930.0        |232.5       |4.35            |
|Bilbao   |4           |875.0        |218.75      |4.5             |
+---------+------------+-------------+------------+----------------+



ingresosPorCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, num_reservas: bigint ... 3 more fields]

## Pregunta 9 — Canal de venta más rentable por ciudad

In [11]:
val ingresosCiudadCanal = reservasConMetricas
  .groupBy("ciudad", "canal")
  .agg(
    round(sum("ingreso_reserva"), 2).alias("ingreso_total"),
    count("*").alias("num_reservas")
  )
  .orderBy($"ciudad", $"ingreso_total".desc)

ingresosCiudadCanal.show(50, truncate = false)

+---------+-------+-------------+------------+
|ciudad   |canal  |ingreso_total|num_reservas|
+---------+-------+-------------+------------+
|Barcelona|Booking|1260.0       |4           |
|Barcelona|Airbnb |360.0        |1           |
|Bilbao   |Directo|680.0        |3           |
|Bilbao   |Airbnb |195.0        |1           |
|Madrid   |Airbnb |1565.0       |4           |
|Madrid   |Directo|640.0        |2           |
|Madrid   |Booking|520.0        |1           |
|Sevilla  |Airbnb |550.0        |2           |
|Sevilla  |Booking|220.0        |1           |
|Sevilla  |Directo|160.0        |1           |
|Valencia |Booking|365.0        |2           |
|Valencia |Directo|360.0        |2           |
|Valencia |Airbnb |225.0        |1           |
+---------+-------+-------------+------------+



ingresosCiudadCanal: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, canal: string ... 2 more fields]

## Pregunta 10 — Ranking de apartamentos dentro de cada ciudad (`Window` + `rank`)

In [12]:
val ingresosPorApt = reservasConMetricas
  .groupBy("id_apartamento", "ciudad")
  .agg(round(sum("ingreso_reserva"), 2).alias("ingreso_total"))

val ventanaCiudad = Window.partitionBy("ciudad").orderBy($"ingreso_total".desc)

val rankingApt = ingresosPorApt
  .withColumn("ranking_ciudad", rank().over(ventanaCiudad))

println("=== Ranking completo por ciudad ===")
rankingApt.orderBy($"ciudad", $"ranking_ciudad").show(truncate = false)

println("=== Top 1 de cada ciudad ===")
rankingApt
  .filter($"ranking_ciudad" === 1)
  .orderBy($"ingreso_total".desc)
  .show(truncate = false)

=== Ranking completo por ciudad ===
+--------------+---------+-------------+--------------+
|id_apartamento|ciudad   |ingreso_total|ranking_ciudad|
+--------------+---------+-------------+--------------+
|APT002        |Barcelona|1080.0       |1             |
|APT007        |Barcelona|540.0        |2             |
|APT005        |Bilbao   |680.0        |1             |
|APT009        |Bilbao   |195.0        |2             |
|APT001        |Madrid   |1425.0       |1             |
|APT006        |Madrid   |1300.0       |2             |
|APT004        |Sevilla  |770.0        |1             |
|APT010        |Sevilla  |160.0        |2             |
|APT003        |Valencia |600.0        |1             |
|APT008        |Valencia |350.0        |2             |
+--------------+---------+-------------+--------------+

=== Top 1 de cada ciudad ===
+--------------+---------+-------------+--------------+
|id_apartamento|ciudad   |ingreso_total|ranking_ciudad|
+--------------+---------+------------

ingresosPorApt: org.apache.spark.sql.package.DataFrame = [id_apartamento: string, ciudad: string ... 1 more field]
ventanaCiudad: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@35836442
rankingApt: org.apache.spark.sql.package.DataFrame = [id_apartamento: string, ciudad: string ... 2 more fields]